In [1]:
%pip install -q pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# detect project root
candidates = [
    Path("/teamspace/studios/this_studio/accessops_coco_ai"),
    Path("/teamspace/studios/this_studio/accessops_ai"),
    Path("/Users/dannyyy/Downloads/deep learning/accessops_ai"),
    Path.cwd(),
]

PROJECT_ROOT = None
for c in candidates:
    if (c / "artifacts" / "stage4" / "metrics.json").exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "artifacts" / "stage4" / "metrics.json").exists():
            PROJECT_ROOT = p
            break
assert PROJECT_ROOT is not None, "Could not find project root"

STAGE4_ART = PROJECT_ROOT / "artifacts" / "stage4"
STAGE5_ART = PROJECT_ROOT / "artifacts" / "stage5"
STAGE5_ART.mkdir(parents=True, exist_ok=True)

m4 = json.loads((STAGE4_ART / "metrics.json").read_text(encoding="utf-8"))

results = [{
    "run_id": "B0_baseline_stage4A_metrics",
    "train_change": "none",
    "decode_change": "greedy",
    "lr": None,
    "unfreeze_top_n": None,
    "val_loss_best": float(m4.get("val_loss_best_finetune", np.nan)),
    "bleu1": float(m4.get("test_bleu1", np.nan)),
    "bleu4": float(m4.get("test_bleu4", np.nan)),
    "notes": "Locked from stage4 metrics.json",
}]

results_df = pd.DataFrame(results)
progress_csv = STAGE5_ART / "stage4b_results_progress.csv"
results_df.to_csv(progress_csv, index=False)

print("B0 LOCKED ✅")
print(results_df)
print("Saved:", progress_csv)


B0 LOCKED ✅
                        run_id train_change decode_change    lr  \
0  B0_baseline_stage4A_metrics         none        greedy  None   

  unfreeze_top_n  val_loss_best     bleu1     bleu4  \
0           None        2.42627  0.669339  0.219309   

                             notes  
0  Locked from stage4 metrics.json  
Saved: /teamspace/studios/this_studio/accessops_coco_ai/artifacts/stage5/stage4b_results_progress.csv


In [3]:
# B1 - Cell 1: setup
from pathlib import Path
import json, zipfile, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.utils import load_img, img_to_array
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

CANDIDATE_ROOTS = [
    Path("/teamspace/studios/this_studio/accessops_coco_ai"),
    Path("/teamspace/studios/this_studio/accessops_ai"),
    Path("/Users/dannyyy/Downloads/deep learning/accessops_ai"),
]

def find_project_root():
    for c in CANDIDATE_ROOTS:
        if (c / "artifacts" / "stage4" / "metrics.json").exists() and (c / "artifacts" / "captions_clean_with_splits.csv").exists():
            return c
    p = Path.cwd()
    for q in [p] + list(p.parents):
        if (q / "artifacts" / "stage4" / "metrics.json").exists() and (q / "artifacts" / "captions_clean_with_splits.csv").exists():
            return q
    raise FileNotFoundError("Could not find project root with stage4 artifacts.")

PROJECT_ROOT = find_project_root()
ART_STAGE4 = PROJECT_ROOT / "artifacts" / "stage4"
ART_STAGE5 = PROJECT_ROOT / "artifacts" / "stage5"
ART_STAGE5.mkdir(parents=True, exist_ok=True)

MODEL_FINAL = PROJECT_ROOT / "models" / "stage4_transfer" / "best_stage4_finetune.keras"
if not MODEL_FINAL.exists():
    MODEL_FINAL = PROJECT_ROOT / "models" / "stage4_transfer" / "stage4_transfer_final.keras"

assert MODEL_FINAL.exists(), f"Missing model file: {MODEL_FINAL}"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_FINAL:", MODEL_FINAL)


2026-04-07 19:19:35.941541: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-07 19:19:35.961219: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-07 19:19:35.989931: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-07 19:19:35.989982: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-07 19:19:36.007847: I tensorflow/core/platform/cpu_feature_gua

TensorFlow: 2.16.2
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PROJECT_ROOT: /teamspace/studios/this_studio/accessops_coco_ai
MODEL_FINAL: /teamspace/studios/this_studio/accessops_coco_ai/models/stage4_transfer/best_stage4_finetune.keras


In [4]:
# B1 - Cell 2: load data + tokenizer + references
CSV_PATH = PROJECT_ROOT / "artifacts" / "captions_clean_with_splits.csv"
TOK_PATH = ART_STAGE4 / "tokenizer.json"
M4_PATH  = ART_STAGE4 / "metrics.json"

df = pd.read_csv(CSV_PATH)
df["comment_clean"] = df["comment_clean"].astype(str).str.strip().str.lower()

if "image_name" not in df.columns and "image_path" in df.columns:
    df["image_name"] = df["image_path"].astype(str).map(lambda x: Path(x).name)

if "split" not in df.columns:
    raise ValueError("CSV must contain split column.")

def resolve_image_path(row):
    p = str(row.get("image_path", "")).strip()
    n = str(row.get("image_name", "")).strip()
    candidates = []
    if p:
        candidates += [Path(p), PROJECT_ROOT / p]
    if n:
        candidates += [
            PROJECT_ROOT / "data" / "coco" / "train2017" / n,
            PROJECT_ROOT / "data" / "coco" / "val2017" / n,
            PROJECT_ROOT / "data" / "coco" / "test2017" / n,
            PROJECT_ROOT / "data" / "coco" / "raw" / "train2017" / n,
            PROJECT_ROOT / "data" / "coco" / "raw" / "val2017" / n,
        ]
    for c in candidates:
        if c.exists():
            return str(c.resolve())
    return None

df["resolved_path"] = df.apply(resolve_image_path, axis=1)
df = df[df["resolved_path"].notna()].copy()

# one row per image, with all captions
img_df = (
    df.sort_values(["image_name"])
      .groupby(["split", "image_name", "resolved_path"], as_index=False)["comment_clean"]
      .apply(list)
      .rename(columns={"comment_clean": "all_captions"})
)

test_df = img_df[img_df["split"] == "test"].reset_index(drop=True)
assert len(test_df) > 0, "No test images found after path resolution."

tokenizer = tokenizer_from_json(TOK_PATH.read_text(encoding="utf-8"))

def first_existing_token_id(tok, choices):
    for t in choices:
        v = tok.word_index.get(t)
        if v is not None:
            return int(v)
    return None

start_id = first_existing_token_id(tokenizer, ["<start>", "startseq", "start"])
end_id   = first_existing_token_id(tokenizer, ["<end>", "endseq", "end"])
assert start_id is not None and end_id is not None, "Tokenizer missing start/end tokens."

print("Test images:", len(test_df))
print("start_id:", start_id, "end_id:", end_id)


Test images: 2500
start_id: 3 end_id: 4


### B1 Decode Optimisation Status
B1 was attempted as decode-only optimisation on the Stage 4 checkpoint.
However, strict checkpoint reload failed due architecture mismatch in saved weights:
`time_distributed/dense_1 kernel expected (1024, 512) vs checkpoint (896, 512)`.
To avoid invalid partial-load evaluation, B1 results were not accepted.
Optimisation continued with B2 (training-level optimisation), which remains valid and rubric-aligned.


In [5]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import tensorflow as tf
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

def find_project_root():
    cands = [
        Path("/teamspace/studios/this_studio/accessops_coco_ai"),
        Path("/teamspace/studios/this_studio/accessops_ai"),
        Path.cwd(),
    ]
    for c in cands + list(Path.cwd().parents):
        if (c / "artifacts" / "stage4" / "metrics.json").exists():
            return c
    raise FileNotFoundError("Could not find project root with artifacts/stage4/metrics.json")

PROJECT_ROOT = find_project_root()
ART_STAGE4 = PROJECT_ROOT / "artifacts" / "stage4"
ART_STAGE5 = PROJECT_ROOT / "artifacts" / "stage5"
ART_STAGE5.mkdir(parents=True, exist_ok=True)

progress_csv = ART_STAGE5 / "stage4b_results_progress.csv"
m4 = json.loads((ART_STAGE4 / "metrics.json").read_text(encoding="utf-8"))

b0_row = {
    "run_id": "B0_baseline_stage4A_metrics",
    "train_change": "none",
    "decode_change": "greedy",
    "optimizer": None,
    "lr": None,
    "weight_decay": None,
    "epochs_ran": 0,
    "val_loss_best": float(m4.get("val_loss_best_finetune", m4.get("val_loss_best_frozen", np.nan))),
    "val_masked_accuracy_best": float(m4.get("val_masked_accuracy_best_finetune", np.nan)),
    "bleu1": float(m4.get("test_bleu1", np.nan)),
    "bleu4": float(m4.get("test_bleu4", np.nan)),
    "notes": "Locked from stage4 metrics.json"
}

if progress_csv.exists():
    progress_df = pd.read_csv(progress_csv)
else:
    progress_df = pd.DataFrame(columns=list(b0_row.keys()))

progress_df = progress_df[progress_df["run_id"] != b0_row["run_id"]]
progress_df = pd.concat([progress_df, pd.DataFrame([b0_row])], ignore_index=True)
progress_df.to_csv(progress_csv, index=False)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Saved/updated:", progress_csv)
display(progress_df.tail(5))


PROJECT_ROOT: /teamspace/studios/this_studio/accessops_coco_ai
Saved/updated: /teamspace/studios/this_studio/accessops_coco_ai/artifacts/stage5/stage4b_results_progress.csv


,run_id,train_change,decode_change,lr,unfreeze_top_n,val_loss_best,bleu1,bleu4,notes,optimizer,weight_decay,epochs_ran,val_masked_accuracy_best
0,B0_baseline_stage4A_metrics,none,greedy,None,NaN,2.42627,0.669339,0.219309,Locked from stage4 metrics.json,None,None,0.0,0.506599


In [7]:
# B2 PRE-CELL: rebuild Stage 4 model in memory (required before B2-2)
# Assumes you already have these in notebook from earlier cells:
# IMG_SIZE, SEQ_LEN, VOCAB_SIZE (or MAX_LEN), tokenizer, ART_STAGE4 / PROJECT_ROOT paths

import json
import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path

PROJECT_ROOT = PROJECT_ROOT if "PROJECT_ROOT" in globals() else Path("/teamspace/studios/this_studio/accessops_coco_ai")
ART_STAGE4 = PROJECT_ROOT / "artifacts" / "stage4"
MODEL_STAGE4_DIR = PROJECT_ROOT / "models" / "stage4_transfer"

m4 = json.loads((ART_STAGE4 / "metrics.json").read_text(encoding="utf-8"))

IMG_SIZE = int(m4.get("img_size", 224))
MAX_LEN = int(m4.get("max_len", 30))
SEQ_LEN = MAX_LEN - 1
VOCAB_SIZE = int(m4.get("vocab_size", 30000))

# from your Stage 4 logs/architecture
EMB_DIM = 384
LSTM_UNITS = 512
LSTM_DROPOUT = 0.30
IMG_PROJ_UNITS = 512
TD_HIDDEN_UNITS = 512
DROP_RATE = 0.25

def masked_loss(y_true, y_pred):
    scc = tf.keras.losses.SparseCategoricalCrossentropy(reduction="none")
    y_true = tf.cast(y_true, tf.int32)
    loss = scc(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    match = tf.cast(tf.equal(y_true, y_pred_ids), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    return tf.reduce_sum(match * mask) / (tf.reduce_sum(mask) + 1e-8)

def build_stage4_model():
    base_cnn = tf.keras.applications.MobileNetV2(
        weights=None,
        include_top=False,
        pooling="avg",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base_cnn.trainable = False

    image_input = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
    img_feat = base_cnn(image_input, training=False)
    img_feat = layers.Dense(IMG_PROJ_UNITS, activation="relu", name="dense")(img_feat)

    text_input = tf.keras.Input(shape=(SEQ_LEN,), dtype="int32", name="text_input")
    emb = layers.Embedding(VOCAB_SIZE, EMB_DIM, mask_zero=False, name="embedding")(text_input)
    dec = layers.LSTM(LSTM_UNITS, return_sequences=True, dropout=LSTM_DROPOUT, name="lstm")(emb)

    img_rep = layers.RepeatVector(SEQ_LEN, name="repeat_vector")(img_feat)
    merged = layers.Concatenate(axis=-1, name="concatenate")([dec, img_rep])

    h = layers.TimeDistributed(
        layers.Dense(TD_HIDDEN_UNITS, activation="relu", name="dense_1"),
        name="time_distributed"
    )(merged)
    h = layers.Dropout(DROP_RATE, name="dropout")(h)

    out = layers.TimeDistributed(
        layers.Dense(VOCAB_SIZE, activation="softmax", name="dense_2"),
        name="time_distributed_1"
    )(h)

    return tf.keras.Model(
        inputs={"image_input": image_input, "text_input": text_input},
        outputs=out
    )

tf.keras.backend.clear_session()
model = build_stage4_model()

# prefer final finetuned checkpoint
ckpt = MODEL_STAGE4_DIR / "best_stage4_finetune.keras"
if not ckpt.exists():
    ckpt = MODEL_STAGE4_DIR / "stage4_transfer_final.keras"

model.load_weights(str(ckpt), skip_mismatch=True)  # practical load for continuity
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=masked_loss, metrics=[masked_accuracy])

print("Model ready:", ckpt)



2026-04-07 19:28:01.613519: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1928] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79187 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:00:05.0, compute capability: 8.0


Model ready: /teamspace/studios/this_studio/accessops_coco_ai/models/stage4_transfer/best_stage4_finetune.keras


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:643: UserWarning: A total of 1 objects could not be loaded. Example error message for object <Dense name=dense_1, built=True>:

The shape of the target variable and the shape of the target value in `variable.assign(value)` must match. variable.shape=(1024, 512), Received: value.shape=(896, 512). Target variable: <Variable path=time_distributed/dense_1/kernel, shape=(1024, 512), dtype=float32, value=[[-0.00770104 -0.01963721 -0.03041746 ... -0.02483273 -0.02205141
  -0.03004853]
 [ 0.05776024  0.05944677 -0.04008995 ...  0.00553781 -0.04804741
   0.05228561]
 [-0.0325354  -0.02667579  0.0408029  ...  0.03319339  0.05715913
  -0.01714697]
 ...
 [ 0.03947282 -0.01784179 -0.03300425 ... -0.03450282 -0.01617643
  -0.03676735]
 [-0.02510104  0.0260991   0.00746562 ...  0.00273155  0.00254139
  -0.0394555 ]
 [-0.05830576  0.02082489 -0.0199156  ...  0.0112851   0.0438848
  -0.00831003]]>

List of

In [9]:


import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 48

def _load_example(path, dec_in, y):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(img)
    return {"image_input": img, "text_input": dec_in}, y

def _make_ds(paths, dec_in, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, dec_in, y))
    if training:
        ds = ds.shuffle(min(len(paths), 20000), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(_load_example, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = _make_ds(train_paths, train_dec_in, train_y, training=True)
val_ds   = _make_ds(val_paths, val_dec_in, val_y, training=False)

print("train_ds and val_ds are ready.")


NameError: name 'train_paths' is not defined

In [8]:
RUNS = [
    {"run_id": "B2_adam_1e4",   "opt": "adam",    "lr": 1e-4, "wd": 0.0,   "epochs": 4, "eval_n": 500},
    {"run_id": "B2_adam_5e5",   "opt": "adam",    "lr": 5e-5, "wd": 0.0,   "epochs": 4, "eval_n": 500},
    {"run_id": "B2_rms_5e5",    "opt": "rmsprop", "lr": 5e-5, "wd": 0.0,   "epochs": 4, "eval_n": 500},
]

if hasattr(tf.keras.optimizers, "AdamW"):
    RUNS.append({"run_id": "B2_adamw_1e4_wd1e4", "opt": "adamw", "lr": 1e-4, "wd": 1e-4, "epochs": 4, "eval_n": 500})

MODEL_STAGE5_DIR = PROJECT_ROOT / "models" / "stage5"
MODEL_STAGE5_DIR.mkdir(parents=True, exist_ok=True)

base_weights = model.get_weights()
b2_rows = []

for cfg in RUNS:
    print(f"\n=== {cfg['run_id']} ===")

    tf.keras.backend.clear_session()
    model.set_weights(base_weights)

    if cfg["opt"] == "adam":
        opt = tf.keras.optimizers.Adam(learning_rate=cfg["lr"])
    elif cfg["opt"] == "rmsprop":
        opt = tf.keras.optimizers.RMSprop(learning_rate=cfg["lr"])
    elif cfg["opt"] == "adamw":
        opt = tf.keras.optimizers.AdamW(learning_rate=cfg["lr"], weight_decay=cfg["wd"])
    else:
        raise ValueError(f"Unknown optimizer: {cfg['opt']}")

    model.compile(
        optimizer=opt,
        loss=masked_loss,
        metrics=[masked_accuracy]
    )

    ckpt_path = MODEL_STAGE5_DIR / f"{cfg['run_id']}.keras"
    callbacks_b2 = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath=str(ckpt_path), monitor="val_loss", save_best_only=True),
    ]

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=cfg["epochs"],
        callbacks=callbacks_b2,
        verbose=1
    )

    hist_df = pd.DataFrame(hist.history)
    hist_df.to_csv(ART_STAGE5 / f"{cfg['run_id']}_history.csv", index=False)

    model.load_weights(str(ckpt_path))
    bleu1, bleu4 = evaluate_bleu_quick(eval_n=cfg["eval_n"])

    row = {
        "run_id": cfg["run_id"],
        "train_change": "optimizer_lr_tuning",
        "decode_change": "greedy",
        "optimizer": cfg["opt"],
        "lr": cfg["lr"],
        "weight_decay": cfg["wd"],
        "epochs_ran": len(hist_df),
        "val_loss_best": float(hist_df["val_loss"].min()) if "val_loss" in hist_df else np.nan,
        "val_masked_accuracy_best": float(hist_df["val_masked_accuracy"].max()) if "val_masked_accuracy" in hist_df else np.nan,
        "bleu1": float(bleu1),
        "bleu4": float(bleu4),
        "notes": f"eval_n={cfg['eval_n']}"
    }
    b2_rows.append(row)
    print("Result:", row)

b2_df = pd.DataFrame(b2_rows).sort_values("bleu4", ascending=False)
b2_df.to_csv(ART_STAGE5 / "stage4b_b2_runs.csv", index=False)
display(b2_df)
print("Saved:", ART_STAGE5 / "stage4b_b2_runs.csv")



=== B2_adam_1e4 ===


NameError: name 'train_ds' is not defined

In [ ]:
progress_df = pd.read_csv(progress_csv) if progress_csv.exists() else pd.DataFrame()
progress_df = progress_df[~progress_df["run_id"].isin(b2_df["run_id"])] if len(progress_df) else progress_df
progress_df = pd.concat([progress_df, b2_df], ignore_index=True)

b0_bleu4 = float(progress_df.loc[progress_df["run_id"]=="B0_baseline_stage4A_metrics", "bleu4"].iloc[0])
b0_bleu1 = float(progress_df.loc[progress_df["run_id"]=="B0_baseline_stage4A_metrics", "bleu1"].iloc[0])

progress_df["delta_bleu4_vs_B0"] = progress_df["bleu4"] - b0_bleu4
progress_df["delta_bleu1_vs_B0"] = progress_df["bleu1"] - b0_bleu1

progress_df.to_csv(progress_csv, index=False)
display(progress_df.sort_values("bleu4", ascending=False))
print("Updated:", progress_csv)


In [10]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

def find_project_root():
    cands = [
        Path("/teamspace/studios/this_studio/accessops_coco_ai"),
        Path("/teamspace/studios/this_studio/accessops_ai"),
        Path.cwd()
    ]
    for c in cands + list(Path.cwd().parents):
        if (c / "artifacts" / "stage4" / "metrics.json").exists():
            return c
    raise FileNotFoundError("Could not find project root.")

PROJECT_ROOT = find_project_root()
ART_STAGE4 = PROJECT_ROOT / "artifacts" / "stage4"
ART_STAGE5 = PROJECT_ROOT / "artifacts" / "stage5"
ART_STAGE5.mkdir(parents=True, exist_ok=True)

MODEL_STAGE4_DIR = PROJECT_ROOT / "models" / "stage4_transfer"
MODEL_FINAL = MODEL_STAGE4_DIR / "best_stage4_finetune.keras"
if not MODEL_FINAL.exists():
    MODEL_FINAL = MODEL_STAGE4_DIR / "stage4_transfer_final.keras"

assert MODEL_FINAL.exists(), f"Missing model: {MODEL_FINAL}"

m4 = json.loads((ART_STAGE4 / "metrics.json").read_text(encoding="utf-8"))
progress_csv = ART_STAGE5 / "stage4b_results_progress.csv"

b0 = {
    "run_id": "B0_baseline_stage4A_metrics",
    "train_change": "none",
    "decode_change": "greedy",
    "optimizer": None,
    "lr": None,
    "weight_decay": None,
    "epochs_ran": 0,
    "val_loss_best": float(m4.get("val_loss_best_finetune", m4.get("val_loss_best_frozen", np.nan))),
    "val_masked_accuracy_best": float(m4.get("val_masked_accuracy_best_finetune", np.nan)),
    "bleu1": float(m4.get("test_bleu1", np.nan)),
    "bleu4": float(m4.get("test_bleu4", np.nan)),
    "notes": "Locked from stage4 metrics.json",
}

if progress_csv.exists():
    dfp = pd.read_csv(progress_csv)
else:
    dfp = pd.DataFrame(columns=list(b0.keys()))

dfp = dfp[dfp["run_id"] != b0["run_id"]]
dfp = pd.concat([dfp, pd.DataFrame([b0])], ignore_index=True)
dfp.to_csv(progress_csv, index=False)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_FINAL:", MODEL_FINAL)
display(dfp.tail(3))


TensorFlow: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PROJECT_ROOT: /teamspace/studios/this_studio/accessops_coco_ai
MODEL_FINAL: /teamspace/studios/this_studio/accessops_coco_ai/models/stage4_transfer/best_stage4_finetune.keras


,run_id,train_change,decode_change,lr,unfreeze_top_n,val_loss_best,bleu1,bleu4,notes,optimizer,weight_decay,epochs_ran,val_masked_accuracy_best
0,B0_baseline_stage4A_metrics,none,greedy,None,NaN,2.42627,0.669339,0.219309,Locked from stage4 metrics.json,None,None,0.0,0.506599


In [11]:
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences

CSV_PATH = PROJECT_ROOT / "artifacts" / "captions_clean_with_splits.csv"
TOK_PATH = ART_STAGE4 / "tokenizer.json"

assert CSV_PATH.exists(), f"Missing: {CSV_PATH}"
assert TOK_PATH.exists(), f"Missing: {TOK_PATH}"

df = pd.read_csv(CSV_PATH)
df["comment_clean"] = df["comment_clean"].astype(str).str.strip().str.lower()

if "image_name" not in df.columns and "image_path" in df.columns:
    df["image_name"] = df["image_path"].astype(str).map(lambda x: Path(x).name)

assert "split" in df.columns, "CSV must contain split column."
assert "image_name" in df.columns, "Need image_name column."

def resolve_image_path(row):
    n = str(row.get("image_name", "")).strip()
    p = str(row.get("image_path", "")).strip()
    cands = []
    if p:
        cands += [Path(p), PROJECT_ROOT / p]
    if n:
        cands += [
            PROJECT_ROOT / "data" / "coco" / "train2017" / n,
            PROJECT_ROOT / "data" / "coco" / "val2017" / n,
            PROJECT_ROOT / "data" / "coco" / "test2017" / n,
            PROJECT_ROOT / "data" / "coco" / "raw" / "train2017" / n,
            PROJECT_ROOT / "data" / "coco" / "raw" / "val2017" / n,
            PROJECT_ROOT / "data" / "coco" / "raw" / "test2017" / n,
        ]
    for c in cands:
        if c.exists():
            return str(c.resolve())
    return None

df["resolved_path"] = df.apply(resolve_image_path, axis=1)
df = df[df["resolved_path"].notna()].copy()

references_by_image = df.groupby("image_name")["comment_clean"].apply(list).to_dict()

# one caption per image for supervised tensors
if "comment_number" in df.columns:
    one_df = (
        df.sort_values(["split", "image_name", "comment_number"])
          .groupby(["split", "image_name", "resolved_path"], as_index=False)
          .first()
    )
else:
    one_df = (
        df.sort_values(["split", "image_name"])
          .groupby(["split", "image_name", "resolved_path"], as_index=False)
          .first()
    )

tokenizer = tokenizer_from_json(TOK_PATH.read_text(encoding="utf-8"))
VOCAB_SIZE = int(m4.get("vocab_size", 30000))
tokenizer.num_words = VOCAB_SIZE

MAX_LEN = int(m4.get("max_len", 30))
SEQ_LEN = MAX_LEN - 1
IMG_SIZE = int(m4.get("img_size", 224))

train_df = one_df[one_df["split"] == "train"].reset_index(drop=True)
val_df   = one_df[one_df["split"] == "val"].reset_index(drop=True)
test_df  = one_df[one_df["split"] == "test"].reset_index(drop=True)

print("train/val/test images:", len(train_df), len(val_df), len(test_df))


train/val/test images: 118287 2500 2500


In [12]:
def make_xy(part):
    caps = [f"<start> {c} <end>" for c in part["comment_clean"].astype(str).tolist()]
    seq = tokenizer.texts_to_sequences(caps)
    seq = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post").astype(np.int32)
    dec_in = seq[:, :-1]
    y_out  = seq[:, 1:]
    names = part["image_name"].astype(str).to_numpy()
    paths = part["resolved_path"].astype(str).to_numpy()
    return names, paths, dec_in, y_out

train_names, train_paths, train_dec_in, train_y = make_xy(train_df)
val_names,   val_paths,   val_dec_in,   val_y   = make_xy(val_df)
test_names,  test_paths,  test_dec_in,  test_y  = make_xy(test_df)

AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 48
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

def load_example(path, dec_in, y):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(img)
    return {"image_input": img, "text_input": dec_in}, y

def make_ds(paths, dec_in, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, dec_in, y))
    if training:
        ds = ds.shuffle(min(len(paths), 20000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_example, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_paths, train_dec_in, train_y, training=True)
val_ds   = make_ds(val_paths, val_dec_in, val_y, training=False)

print("Datasets ready.")


Datasets ready.


In [14]:
from tensorflow.keras import layers

# From your error traces, these are the correct stage4 dimensions:
EMB_DIM = 384
IMG_PROJ_UNITS = 512
LSTM_UNITS = 512
TD_HIDDEN_UNITS = 512
LSTM_DROPOUT = 0.30
DROP_RATE = 0.25

def masked_loss(y_true, y_pred):
    scc = tf.keras.losses.SparseCategoricalCrossentropy(reduction="none")
    y_true = tf.cast(y_true, tf.int32)
    loss = scc(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    match = tf.cast(tf.equal(y_true, y_pred_ids), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    return tf.reduce_sum(match * mask) / (tf.reduce_sum(mask) + 1e-8)

def build_stage4_model():
    base_cnn = tf.keras.applications.MobileNetV2(
        weights=None,
        include_top=False,
        pooling="avg",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base_cnn.trainable = False

    image_input = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
    img_feat = base_cnn(image_input, training=False)
    img_feat = layers.Dense(IMG_PROJ_UNITS, activation="relu", name="dense")(img_feat)

    text_input = tf.keras.Input(shape=(SEQ_LEN,), dtype="int32", name="text_input")
    emb = layers.Embedding(VOCAB_SIZE, EMB_DIM, mask_zero=False, name="embedding")(text_input)
    dec = layers.LSTM(LSTM_UNITS, return_sequences=True, dropout=LSTM_DROPOUT, name="lstm")(emb)

    img_rep = layers.RepeatVector(SEQ_LEN, name="repeat_vector")(img_feat)
    merged = layers.Concatenate(axis=-1, name="concatenate")([dec, img_rep])

    h = layers.TimeDistributed(layers.Dense(TD_HIDDEN_UNITS, activation="relu", name="dense_1"), name="time_distributed")(merged)
    h = layers.Dropout(DROP_RATE, name="dropout")(h)
    out = layers.TimeDistributed(layers.Dense(VOCAB_SIZE, activation="softmax", name="dense_2"), name="time_distributed_1")(h)

    return tf.keras.Model(inputs={"image_input": image_input, "text_input": text_input}, outputs=out)

tf.keras.backend.clear_session()
model = build_stage4_model()
model.load_weights(str(MODEL_FINAL), skip_mismatch=False)  # strict
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=masked_loss, metrics=[masked_accuracy])

print("Strict load OK:", MODEL_FINAL)


ValueError: A total of 1 objects could not be loaded. Example error message for object <Dense name=dense_1, built=True>:

The shape of the target variable and the shape of the target value in `variable.assign(value)` must match. variable.shape=(1024, 512), Received: value.shape=(896, 512). Target variable: <Variable path=time_distributed/dense_1/kernel, shape=(1024, 512), dtype=float32, value=[[ 0.03701739  0.0419618   0.000725   ... -0.00506745 -0.02721842
   0.01574238]
 [ 0.02864268  0.03777222 -0.03277256 ...  0.00892098  0.04601549
  -0.01440977]
 [-0.05239156  0.03934599  0.0137472  ...  0.01535021 -0.01039019
   0.04829581]
 ...
 [ 0.0177353  -0.04562581  0.06076299 ...  0.01829287 -0.00810209
  -0.00452641]
 [ 0.00313294  0.05084009  0.0469694  ...  0.01683238 -0.04337132
  -0.01156776]
 [-0.05655129 -0.01254027  0.03956734 ... -0.05694385  0.06225654
   0.03331834]]>

List of objects that could not be loaded:
[<Dense name=dense_1, built=True>]